# Booze ’R’ Us — City-Category-Month Data Build

**Client:** Booze ’R’ Us — projecting sales for the coming year and helping
adjust inventory and sales tactics across Iowa’s major city markets.

**Observational unit:** city × alcohol category × year-month, limited to the
ten Iowa cities with the highest total liquor volume.

**Target:** total_bottles, which represents the net number of bottles
ordered by retailers in a city-category-month. We also keep total liters and
total sales dollars for descriptive analysis. For the linear regression model,
we create log1p_total_bottles so that large sales values and zero-sales
months can be handled appropriately.

This notebook builds city_category_month.parquet from the raw
liquor_2022_2026.parquet. The final dataset is used
in the Booze ’R’ Us analysis notebook to forecast bottle demand by city and
alcohol category.

# Step 1 — Load and Inspect the Raw Data

We begin by loading only the columns needed for this analysis: transaction
date, retailer city, alcohol category, bottles sold, liters sold, and sales
dollars. Limiting the columns helps us work more efficiently with the large
raw dataset.

The raw file contains individual transaction line records. Before aggregating
them, we inspect the data types, missing values, negative sales values, and
available date range so that we understand any data quality issues that could
affect the model.

In [1]:
import pandas as pd
import numpy as np

file_path = "liquor_2022_2026.parquet"

# Only load the columns needed to build the city-category-month dataset
needed_columns = [
    "ordered_on",
    "store_city",
    "category_name",
    "sales_bottles",
    "sales_liters",
    "sales_dollars"
]

raw_df = pd.read_parquet(
    file_path,
    columns=needed_columns
)

print("Rows loaded:", f"{len(raw_df):,}")
print("\nColumn types:")
print(raw_df.dtypes)

raw_df.head()

Rows loaded: 11,889,904

Column types:
ordered_on        object
store_city           str
category_name        str
sales_bottles        str
sales_liters     float64
sales_dollars    float64
dtype: object


,ordered_on,store_city,category_name,sales_bottles,sales_liters,sales_dollars
0,2022-11-08,MARSHALLTOWN,IMPORTED BRANDIES,24,4.80,269.76
1,2022-08-02,DUBUQUE,AMERICAN VODKAS,12,12.00,71.52
2,2022-08-02,MISSOURI VALLEY,STRAIGHT BOURBON WHISKIES,3,2.25,49.50
3,2022-08-04,CLEAR LAKE,CANADIAN WHISKIES,30,52.50,470.40
4,2022-09-16,DEWITT,WHISKEY LIQUEUR,1,0.05,38.70


# Step 2 — Clean the Data and Create a Monthly Date Variable

ordered_on is converted to a date, and sales_bottles is converted from
text to a numeric value. We then remove records missing a date, city, or
alcohol category because those records cannot be assigned to a
city-category-month observation.

City and category names are standardized by removing extra spaces and using
uppercase text. This prevents the same city or category from being split into
multiple groups because of inconsistent spelling or formatting.

Negative sales values are kept rather than automatically deleted. They may
represent returns, refunds, or corrections. When the transactions are summed
at the monthly level, these values reduce the total appropriately and produce
a net sales measure.

Finally, we create a monthly date variable that represents the first day of
each month. This is only a consistent label for the year-month period; it does
not mean that every sale occurred on the first day of the month.

In [2]:
# Convert fields to the correct data types
raw_df["ordered_on"] = pd.to_datetime(
    raw_df["ordered_on"],
    errors="coerce"
)

raw_df["sales_bottles"] = pd.to_numeric(
    raw_df["sales_bottles"],
    errors="coerce"
)

# Check whether conversion created missing values
print("Missing values after conversion:")
print(
    raw_df[
        ["ordered_on", "store_city", "category_name",
         "sales_bottles", "sales_liters", "sales_dollars"]
    ].isna().sum()
)

# Check for return/correction records with negative sales
sales_columns = ["sales_bottles", "sales_liters", "sales_dollars"]

print("\nNegative values:")
print((raw_df[sales_columns] < 0).sum())

print("\nAvailable date range:")
print("Earliest:", raw_df["ordered_on"].min())
print("Latest:", raw_df["ordered_on"].max())

Missing values after conversion:
ordered_on          0
store_city       3929
category_name       0
sales_bottles       0
sales_liters        0
sales_dollars       0
dtype: int64

Negative values:
sales_bottles    11391
sales_liters     11207
sales_dollars    11391
dtype: int64

Available date range:
Earliest: 2022-01-02 00:00:00
Latest: 2026-08-31 00:00:00


In [3]:
# Remove records that cannot be used for city-category-month modeling
starting_rows = len(raw_df)

raw_df.dropna(
    subset=["ordered_on", "store_city", "category_name"],
    inplace=True
)

# Standardize text values for consistent grouping
raw_df["store_city"] = raw_df["store_city"].str.strip().str.upper()
raw_df["category_name"] = raw_df["category_name"].str.strip().str.upper()

# Create a monthly date variable for aggregation
raw_df["month"] = (
    raw_df["ordered_on"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

print("Rows removed because city/category/date was missing:",
      f"{starting_rows - len(raw_df):,}")

print("Rows remaining:", f"{len(raw_df):,}")
print("Number of cities:", raw_df["store_city"].nunique())
print("Number of categories:", raw_df["category_name"].nunique())

raw_df[
    ["ordered_on", "month", "store_city", "category_name",
     "sales_bottles", "sales_liters", "sales_dollars"]
].head()

Rows removed because city/category/date was missing: 3,929
Rows remaining: 11,885,975
Number of cities: 487


Number of categories: 52


,ordered_on,month,store_city,category_name,sales_bottles,sales_liters,sales_dollars
0,2022-11-08,2022-11-01,MARSHALLTOWN,IMPORTED BRANDIES,24,4.80,269.76
1,2022-08-02,2022-08-01,DUBUQUE,AMERICAN VODKAS,12,12.00,71.52
2,2022-08-02,2022-08-01,MISSOURI VALLEY,STRAIGHT BOURBON WHISKIES,3,2.25,49.50
3,2022-08-04,2022-08-01,CLEAR LAKE,CANADIAN WHISKIES,30,52.50,470.40
4,2022-09-16,2022-09-01,DEWITT,WHISKEY LIQUEUR,1,0.05,38.70


# Step 3 — Identify the Ten Largest Iowa City Markets

Booze ’R’ Us is interested in adjusting sales tactics by location. To focus
the analysis on the largest markets, we identify the ten Iowa cities with the
highest total liquor volume, measured in liters, across the available data.

Using liters to select the cities gives a more standardized measure of volume
than bottle count because bottle sizes can differ. The remaining analysis is
limited to these ten cities so that the final model is useful for the largest
markets while still being manageable and interpretable.

In [4]:
# Find the ten Iowa cities with the highest total liquor volume
top_cities = (
    raw_df.groupby("store_city", as_index=False)
    .agg(
        total_liters=("sales_liters", "sum"),
        total_bottles=("sales_bottles", "sum"),
        total_sales_dollars=("sales_dollars", "sum")
    )
    .sort_values("total_liters", ascending=False)
    .head(10)
)

# Save their names as a reusable list
major_cities = top_cities["store_city"].tolist()

top_cities

,store_city,total_liters,total_bottles,total_sales_dollars
117,DES MOINES,1.162025e+07,17646051,2.414823e+08
70,CEDAR RAPIDS,6.716216e+06,9403111,1.283975e+08
107,DAVENPORT,5.001526e+06,7641252,9.370377e+07
467,WEST DES MOINES,4.191451e+06,4988338,8.849153e+07
99,COUNCIL BLUFFS,3.503537e+06,5275316,6.993668e+07
401,SIOUX CITY,3.397058e+06,4758853,6.538258e+07
218,IOWA CITY,3.169398e+06,4120812,5.937999e+07
23,ANKENY,3.067312e+06,3549856,5.994442e+07
453,WATERLOO,3.018901e+06,4975068,5.830755e+07
124,DUBUQUE,2.808635e+06,3624298,4.941545e+07


# Step 4 — Aggregate Transactions to City × Category × Year-Month

The raw data contains millions of transaction line records, which is too
detailed for the monthly sales forecasting question. We aggregate those
records so that each observed row represents total liquor orders for one
alcohol category in one city during one month.

For each city-category-month combination, we calculate:
* total_bottles: net bottles ordered after returns or corrections
* total_liters: net liters ordered
* total_sales_dollars: net sales dollars
* line_items: number of original transaction line records contributing to
the monthly total

line_items is useful for checking the aggregation but will not be kept as a
forecasting feature because it would not be known before future sales occur.

In [5]:
# Keep only the ten major city markets
major_city_df = raw_df[
    raw_df["store_city"].isin(major_cities)
].copy()

# Aggregate retailer orders to city-category-month level
city_category_month_observed = (
    major_city_df.groupby(
        ["store_city", "category_name", "month"],
        as_index=False
    )
    .agg(
        total_bottles=("sales_bottles", "sum"),
        total_liters=("sales_liters", "sum"),
        total_sales_dollars=("sales_dollars", "sum"),
        line_items=("sales_bottles", "size")
    )
)

print("Rows in city-category-month dataset:",
      f"{len(city_category_month_observed):,}")

city_category_month_observed.head(10)

Rows in city-category-month dataset: 24,300


,store_city,category_name,month,total_bottles,total_liters,total_sales_dollars,line_items
0,ANKENY,,2026-06-01,180,135.00,8098.20,2
1,ANKENY,100% AGAVE TEQUILA,2022-01-01,667,491.96,18621.92,100
2,ANKENY,100% AGAVE TEQUILA,2022-02-01,1856,1385.21,62033.51,208
3,ANKENY,100% AGAVE TEQUILA,2022-03-01,1283,922.21,36144.98,203
4,ANKENY,100% AGAVE TEQUILA,2022-04-01,2846,2756.69,75998.45,235
5,ANKENY,100% AGAVE TEQUILA,2022-05-01,2724,2842.08,69427.38,268
6,ANKENY,100% AGAVE TEQUILA,2022-06-01,1820,1311.55,58001.64,261
7,ANKENY,100% AGAVE TEQUILA,2022-07-01,1490,1107.62,44850.44,219
8,ANKENY,100% AGAVE TEQUILA,2022-08-01,2334,1882.49,70736.83,289
9,ANKENY,100% AGAVE TEQUILA,2022-09-01,1386,1021.29,41321.49,202


# Step 5 — Create a Complete City-Category-Month Panel

Some city-category-month combinations do not appear in the transaction data.
A missing record can mean that no order was recorded for that category in that
city during that month. It does not necessarily mean the data are missing.

To represent those months correctly, we create every possible combination of
the ten cities, alcohol categories, and months in the available date range.
We merge the observed monthly totals onto this complete grid and fill
unobserved combinations with zero.

This gives the model a consistent panel of data, including zero sales months,
instead of allowing missing rows to be mistaken for missing information. Blank
category labels are removed because they do not provide a meaningful alcohol
category for modeling or sales recommendations.

In [6]:
# Remove blank category labels from the aggregated data
city_category_month_observed = city_category_month_observed[
    city_category_month_observed["category_name"].str.strip().ne("")
].copy()

# Create every possible city-category-month combination
all_months = pd.date_range(
    city_category_month_observed["month"].min(),
    city_category_month_observed["month"].max(),
    freq="MS"
)

all_categories = sorted(
    city_category_month_observed["category_name"].unique()
)

full_grid = pd.MultiIndex.from_product(
    [major_cities, all_categories, all_months],
    names=["store_city", "category_name", "month"]
).to_frame(index=False)

# Merge observed monthly sales onto the complete grid
city_category_month = full_grid.merge(
    city_category_month_observed,
    on=["store_city", "category_name", "month"],
    how="left"
)

# A missing combination means no recorded order that month, so fill with zero
sales_summary_columns = [
    "total_bottles",
    "total_liters",
    "total_sales_dollars",
    "line_items"
]

city_category_month[sales_summary_columns] = (
    city_category_month[sales_summary_columns].fillna(0)
)

print("Final rows:", f"{len(city_category_month):,}")
print("Number of zero-sales city-category-month rows:",
      f"{(city_category_month['total_bottles'] == 0).sum():,}")

city_category_month.head(10)

Final rows: 28,000
Number of zero-sales city-category-month rows: 3,705


,store_city,category_name,month,total_bottles,total_liters,total_sales_dollars,line_items
0,DES MOINES,100% AGAVE TEQUILA,2022-01-01,8791.0,6344.89,295374.91,781.0
1,DES MOINES,100% AGAVE TEQUILA,2022-02-01,12713.0,8953.37,405403.96,915.0
2,DES MOINES,100% AGAVE TEQUILA,2022-03-01,13353.0,9813.03,429574.19,1000.0
3,DES MOINES,100% AGAVE TEQUILA,2022-04-01,16944.0,12028.55,517714.72,1215.0
4,DES MOINES,100% AGAVE TEQUILA,2022-05-01,18813.0,13710.51,592772.69,1232.0
5,DES MOINES,100% AGAVE TEQUILA,2022-06-01,17462.0,12213.67,535398.19,1326.0
6,DES MOINES,100% AGAVE TEQUILA,2022-07-01,17613.0,12579.15,558757.13,1418.0
7,DES MOINES,100% AGAVE TEQUILA,2022-08-01,18668.0,13232.02,595438.62,1363.0
8,DES MOINES,100% AGAVE TEQUILA,2022-09-01,17122.0,11740.10,534729.07,1399.0
9,DES MOINES,100% AGAVE TEQUILA,2022-10-01,16708.0,11876.33,538844.49,1275.0


# Step 6 — Create Date Derived Forecasting Features

The monthly date allows us to create features that can be known before a
future month begins:
* year: calendar year
* month_number: month from 1 through 12, used to capture recurring seasonal
patterns
* time_index: number of months since the beginning of the dataset, used to
estimate an overall time trend
* log1p_total_bottles: the natural log of one plus total bottles

The log transformation reduces the influence of very large sales values and
allows zero sales months to remain in the model because log(1 + 0) = 0.
Predictions can later be converted back into estimated bottle counts.

In [7]:
# Sort the data in time order
city_category_month = city_category_month.sort_values(
    ["store_city", "category_name", "month"]
).reset_index(drop=True)

# Calendar features
city_category_month["year"] = city_category_month["month"].dt.year
city_category_month["month_number"] = city_category_month["month"].dt.month

# Sequential time trend: 0 for the first month, then 1, 2, 3, ...
first_month = city_category_month["month"].min()

city_category_month["time_index"] = (
    (city_category_month["month"].dt.year - first_month.year) * 12
    + (city_category_month["month"].dt.month - first_month.month)
)

# Target for the linear regression
# log1p handles zero-sales months because log(1 + 0) = 0
city_category_month["log1p_total_bottles"] = np.log1p(
    city_category_month["total_bottles"]
)

print(city_category_month.shape)
city_category_month.head(10)

(28000, 11)


,store_city,category_name,month,total_bottles,total_liters,total_sales_dollars,line_items,year,month_number,time_index,log1p_total_bottles
0,ANKENY,100% AGAVE TEQUILA,2022-01-01,667.0,491.96,18621.92,100.0,2022,1,0,6.504288
1,ANKENY,100% AGAVE TEQUILA,2022-02-01,1856.0,1385.21,62033.51,208.0,2022,2,1,7.526718
2,ANKENY,100% AGAVE TEQUILA,2022-03-01,1283.0,922.21,36144.98,203.0,2022,3,2,7.157735
3,ANKENY,100% AGAVE TEQUILA,2022-04-01,2846.0,2756.69,75998.45,235.0,2022,4,3,7.954021
4,ANKENY,100% AGAVE TEQUILA,2022-05-01,2724.0,2842.08,69427.38,268.0,2022,5,4,7.910224
5,ANKENY,100% AGAVE TEQUILA,2022-06-01,1820.0,1311.55,58001.64,261.0,2022,6,5,7.507141
6,ANKENY,100% AGAVE TEQUILA,2022-07-01,1490.0,1107.62,44850.44,219.0,2022,7,6,7.307202
7,ANKENY,100% AGAVE TEQUILA,2022-08-01,2334.0,1882.49,70736.83,289.0,2022,8,7,7.755767
8,ANKENY,100% AGAVE TEQUILA,2022-09-01,1386.0,1021.29,41321.49,202.0,2022,9,8,7.234898
9,ANKENY,100% AGAVE TEQUILA,2022-10-01,2658.0,2626.44,77841.58,248.0,2022,10,9,7.885705


# Step 7 — Finalize and Save the Modeling Dataset

Before saving, we remove line_items because it is calculated from the same
month’s transactions and would not be available when forecasting future
sales. We also rename the monthly date variable to month_start to make its
purpose clearer.

The completed dataset is saved as city_category_month.parquet.

In [8]:
city_category_month = city_category_month.drop(
    columns=["line_items"]
)

city_category_month = city_category_month.rename(
    columns={"month": "month_start"}
)

city_category_month.head()

,store_city,category_name,month_start,total_bottles,total_liters,total_sales_dollars,year,month_number,time_index,log1p_total_bottles
0,ANKENY,100% AGAVE TEQUILA,2022-01-01,667.0,491.96,18621.92,2022,1,0,6.504288
1,ANKENY,100% AGAVE TEQUILA,2022-02-01,1856.0,1385.21,62033.51,2022,2,1,7.526718
2,ANKENY,100% AGAVE TEQUILA,2022-03-01,1283.0,922.21,36144.98,2022,3,2,7.157735
3,ANKENY,100% AGAVE TEQUILA,2022-04-01,2846.0,2756.69,75998.45,2022,4,3,7.954021
4,ANKENY,100% AGAVE TEQUILA,2022-05-01,2724.0,2842.08,69427.38,2022,5,4,7.910224


In [9]:
import os

output_path = "city_category_month.parquet"

# Save the finished modeling dataset
city_category_month.to_parquet(
    output_path,
    index=False
)

# Verify that it saved correctly
saved_df = pd.read_parquet(output_path)

print("Saved file:", output_path)
print("Rows and columns:", saved_df.shape)
print("File size (MB):", round(os.path.getsize(output_path) / 1_000_000, 2))

saved_df.head()

Saved file: city_category_month.parquet
Rows and columns: (28000, 10)
File size (MB): 0.5


,store_city,category_name,month_start,total_bottles,total_liters,total_sales_dollars,year,month_number,time_index,log1p_total_bottles
0,ANKENY,100% AGAVE TEQUILA,2022-01-01,667.0,491.96,18621.92,2022,1,0,6.504288
1,ANKENY,100% AGAVE TEQUILA,2022-02-01,1856.0,1385.21,62033.51,2022,2,1,7.526718
2,ANKENY,100% AGAVE TEQUILA,2022-03-01,1283.0,922.21,36144.98,2022,3,2,7.157735
3,ANKENY,100% AGAVE TEQUILA,2022-04-01,2846.0,2756.69,75998.45,2022,4,3,7.954021
4,ANKENY,100% AGAVE TEQUILA,2022-05-01,2724.0,2842.08,69427.38,2022,5,4,7.910224
